In [ ]:
# Install core multimodal processing and vector database packages
!pip install -q transformers qdrant-client pillow tqdm torch torchvision minio timm sentencepiece torchscale sentence-transformers tensorboardX torchmetrics

# Clone official repo BEiT-3 của Microsoft để lấy architecture
!rm -rf unilm
!git clone https://github.com/microsoft/unilm.git
%cd unilm/beit3

# Tải weights BEiT-3 Multimodal (1024 chiều)
!wget -q https://conversationhub.blob.core.windows.net/beit-share-cn/beit3/beit3_large_patch16_224.pth
!wget -q https://conversationhub.blob.core.windows.net/beit-share-cn/beit3/beit3.spm

%cd /kaggle/working

In [ ]:
import sys
import os
import types
import math
import torch
import urllib.request
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoProcessor
from sentence_transformers import SentenceTransformer

# ==============================================================================
# 1. FIX TOÀN BỘ TƯƠNG THÍCH PYTORCH 2.x & BYPASS THƯ VIỆN TRAINING
# ==============================================================================
if "torch._six" not in sys.modules:
    six_mod = types.ModuleType("torch._six")
    six_mod.inf = math.inf
    six_mod.string_classes = (str, bytes)
    sys.modules["torch._six"] = six_mod

if "tensorboardX" not in sys.modules:
    tbx_mod = types.ModuleType("tensorboardX")
    tbx_mod.SummaryWriter = object
    sys.modules["tensorboardX"] = tbx_mod

if "torchmetrics" not in sys.modules:
    tm_mod = types.ModuleType("torchmetrics")
    tm_mod.Metric = object
    sys.modules["torchmetrics"] = tm_mod

for p in ["/kaggle/working/unilm/beit3", "/kaggle/working/unilm"]:
    if p not in sys.path:
        sys.path.append(p)

import utils
from modeling_utils import BEiT3Wrapper, _get_large_config

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running models on: {device.upper()} (⚡ FP16 Mode Enabled)")

# ==============================================================================
# 2. NẠP SigLIP (1152d) Ở CHẾ ĐỘ FP16
# ==============================================================================
print("Loading SigLIP (FP16)...")
siglip_name = "google/siglip-so400m-patch14-384"
siglip_processor = AutoProcessor.from_pretrained(siglip_name)
siglip_model = AutoModel.from_pretrained(siglip_name, torch_dtype=torch.float16).to(device).eval()

# ==============================================================================
# 3. NẠP BEiT-3 Multimodal (1024d) Ở CHẾ ĐỘ FP16
# ==============================================================================
print("Loading BEiT-3 Multimodal (FP16)...")
config = _get_large_config(img_size=224)
beit_model = BEiT3Wrapper(args=config, is_vision=True)

ckpt_path = "/kaggle/working/beit3_large_patch16_224.pth"
if not os.path.exists(ckpt_path):
    print("-> Downloading BEiT-3 Multimodal weights...")
    try:
        from huggingface_hub import hf_hub_download
        ckpt_path = hf_hub_download(repo_id="timm/beit3_large_patch16_224.pt", filename="pytorch_model.bin")
    except Exception:
        url = "https://huggingface.co/timm/beit3_large_patch16_224.pt/resolve/main/pytorch_model.bin"
        urllib.request.urlretrieve(url, ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu")
state_dict = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt
beit_model.load_state_dict(state_dict, strict=False)
beit_model = beit_model.to(device).half().eval()  # Ép về float16

beit_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ==============================================================================
# 4. NẠP Qwen3-VL-Embedding-2B Ở CHẾ ĐỘ FP16
# ==============================================================================
print("Loading Qwen3-VL-Embedding-2B (FP16)...")
qwen_model = SentenceTransformer(
    "Qwen/Qwen3-VL-Embedding-2B",
    device=device,
    model_kwargs={"torch_dtype": torch.float16}
)
qwen_dim = qwen_model.get_embedding_dimension()
print(f"-> All 3 Models Loaded in FP16: SigLIP (1152d), BEiT-3 (1024d), Qwen ({qwen_dim}d)")

# ==============================================================================
# 5. HÀM TRÍCH XUẤT 3 VECTORS ĐỒNG THỜI VỚI FP16 TENSOR CORES
# ==============================================================================
def _extract_tensor(out):
    """Bóc tách Tensor an toàn."""
    if isinstance(out, torch.Tensor):
        t = out
    elif isinstance(out, dict):
        t = out.get("encoder_out", out.get("pooler_output", out.get("last_hidden_state", list(out.values())[0])))
    elif hasattr(out, "pooler_output") and out.pooler_output is not None:
        t = out.pooler_output
    elif hasattr(out, "image_embeds") and out.image_embeds is not None:
        t = out.image_embeds
    elif hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        t = out.last_hidden_state
    elif isinstance(out, (tuple, list)):
        t = out[0]
    else:
        t = out
        
    if isinstance(t, torch.Tensor) and t.dim() == 3:
        t = t[:, 0]
    return t

def extract_batch_embeddings(images):
    """Trích xuất đồng thời 3 embeddings bằng FP16 siêu tốc."""
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            # A. SigLIP (1152d)
            siglip_inputs = siglip_processor(images=images, return_tensors="pt").to(device)
            siglip_raw = siglip_model.get_image_features(**siglip_inputs)
            siglip_tensor = _extract_tensor(siglip_raw)
            siglip_embs = torch.nn.functional.normalize(siglip_tensor, p=2, dim=-1).float().cpu().numpy().tolist()

            # B. BEiT-3 Multimodal (1024d)
            beit_tensors = torch.stack([beit_transform(img) for img in images]).to(device).half()
            if hasattr(beit_model, "beit3"):
                beit_raw = beit_model.beit3(visual_tokens=beit_tensors)
            else:
                beit_raw = beit_model(beit_tensors)
            beit_tensor = _extract_tensor(beit_raw)
            beit_embs = torch.nn.functional.normalize(beit_tensor, p=2, dim=-1).float().cpu().numpy().tolist()

            # C. Qwen3-VL (2048d)
            qwen_embs = qwen_model.encode(
                images,
                batch_size=len(images),
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False
            ).tolist()

    return siglip_embs, beit_embs, qwen_embs


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models

# --- REPLACEMENT ZONE: INPUT YOUR CLUSTER DETAILS HERE ---
QDRANT_URL = "https://qdrant.viettech.fit/"
QDRANT_API_KEY = "ai20k_tHMKPd49T1_ggY2lQDXcnGEklENw_lnb"
# ---------------------------------------------------------
client = QdrantClient(
    url=QDRANT_URL,
    port=443,
    api_key=QDRANT_API_KEY,
    prefer_grpc=False,
    timeout=180,
)
COLLECTION_NAME = "video_keyframes_2"

print(f"Checking collection status for: '{COLLECTION_NAME}'...")

if not client.collection_exists(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' does not exist. Creating with 3 vectors: siglip, beit3, qwen...")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "siglip": models.VectorParams(size=1152, distance=models.Distance.COSINE),
            "beit3": models.VectorParams(size=1024, distance=models.Distance.COSINE),
            "qwen": models.VectorParams(size=qwen_dim, distance=models.Distance.COSINE)
        }
    )
    print("Collection created successfully with 3 vector spaces.")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists. Appending to existing collection.")

info = client.get_collection(collection_name=COLLECTION_NAME)
print(f"-> Status: {info.status}")
print(f"-> Current Point Count: {info.points_count}")


In [ ]:
import os
import uuid
import gc
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==============================================================================
# ĐƯỜNG DẪN DATASET TRÊN KAGGLE (Bạn chỉnh lại đúng folder Dataset của bạn)
# ==============================================================================
DATASET_ROOT_DIR = "/kaggle/input/datasets/xuanhuydinh/aic-2026-keyframes"  # <-- Đổi theo đường dẫn Dataset của bạn

def fetch_single_file(task):
    """Đọc trực tiếp ảnh từ ổ đĩa Kaggle."""
    try:
        with Image.open(task["path"]) as img:
            return img.convert("RGB"), task
    except Exception as e:
        tqdm.write(f"[Error] Failed reading {task['path']}: {e}")
        return None, None

def process_batch(tasks):
    """Xử lý 1 batch: đọc ảnh disk -> trích xuất 3 vector -> upsert Qdrant."""
    images, valid_tasks = [], []
    
    # 1. Đọc ảnh song song từ đĩa
    with ThreadPoolExecutor(max_workers=8) as executor:
        for img, task in executor.map(fetch_single_file, tasks):
            if img is not None:
                images.append(img)
                valid_tasks.append(task)
                
    if not images:
        return

    # 2. Extract Embeddings 3 mô hình trên GPU
    try:
        siglip_embs, beit_embs, qwen_embs = extract_batch_embeddings(images)
    except Exception as e:
        tqdm.write(f"[Error] CUDA Batch Extraction failed: {e}")
        return
    finally:
        del images
        gc.collect()
        torch.cuda.empty_cache()

    # 3. Tạo Points chứa cả 3 vectors (GIỮ NGUYÊN 100% LOGIC CỦA BẠN)
    points_to_upsert = []
    for task, siglip_vec, beit_vec, qwen_vec in zip(valid_tasks, siglip_embs, beit_embs, qwen_embs):
        frame_filename = task["frame_filename"]
        video_name = task["video_name"]

        # LOGIC GỐC CHUẨN XÁC CỦA BẠN:
        parts = frame_filename.replace(".jpg", "").split("_")
        frame_id = int(parts[3])

        payload = {
            "video_name": video_name,
            "frame_id": frame_id
        }

        point = models.PointStruct(
            id=str(uuid.uuid4()),
            vector={
                "siglip": siglip_vec,
                "beit3": beit_vec,
                "qwen": qwen_vec
            },
            payload=payload
        )
        points_to_upsert.append(point)

    # 4. Upsert vào Qdrant
    client.upsert(collection_name=COLLECTION_NAME, points=points_to_upsert)


def run_kaggle_ingestion(dataset_dir, batch_size=32, target_folders=None):
    """Quét cực nhanh các folder mục tiêu và chạy pipeline."""
    print(f"Scanning target folders in: '{dataset_dir}'...")
    all_tasks = []
    video_names = set()

    # 1. NẾU CÓ TARGET_FOLDERS: Quét thẳng vào từng folder (Mất 0.05s)
    if target_folders is not None:
        for folder_name in target_folders:
            # Tìm đường dẫn của folder video (xử lý cả trường hợp có thư mục con)
            folder_path = os.path.join(dataset_dir, folder_name)
            
            # Nếu folder nằm trong subfolder (như keyframes/batch_1/...)
            if not os.path.exists(folder_path):
                # Tìm nhanh folder
                matched = [os.path.join(root, folder_name) for root, dirs, _ in os.walk(dataset_dir, topdown=True) if folder_name in dirs]
                if matched:
                    folder_path = matched[0]

            if os.path.exists(folder_path):
                video_names.add(folder_name)
                for filename in os.listdir(folder_path):
                    if filename.lower().endswith(".jpg") and filename.startswith("scene_"):
                        all_tasks.append({
                            "video_name": folder_name,
                            "frame_filename": filename,
                            "path": os.path.join(folder_path, filename)
                        })

    # 2. NẾU QUÉT TOÀN BỘ DATASET (Không lọc)
    else:
        for root, _, files in os.walk(dataset_dir):
            for filename in files:
                if filename.lower().endswith(".jpg") and filename.startswith("scene_"):
                    video_name = os.path.basename(root)
                    video_names.add(video_name)
                    all_tasks.append({
                        "video_name": video_name,
                        "frame_filename": filename,
                        "path": os.path.join(root, filename)
                    })

    total_frames = len(all_tasks)
    print(f"Found {len(video_names)} videos with {total_frames:,} total keyframes.")
    if total_frames == 0:
        print("No matching frames found. Please check dataset path!")
        return

    # 3. Gom nhóm theo Batch và xử lý
    batch_tasks = []
    with tqdm(total=total_frames, desc="Indexing Frames", unit="frame") as pbar:
        for task in all_tasks:
            batch_tasks.append(task)
            if len(batch_tasks) >= batch_size:
                process_batch(batch_tasks)
                pbar.update(len(batch_tasks))
                batch_tasks = []
                
        if batch_tasks:
            process_batch(batch_tasks)
            pbar.update(len(batch_tasks))

    print("\nPipeline complete. Database synchronized with 3 vector spaces.")


# Execute
run_kaggle_ingestion(
    dataset_dir=DATASET_ROOT_DIR,
    batch_size=16,
    target_folders=[f"L21_V{i:03d}" for i in range(7, 32)] + [f"L22_V{i:03d}" for i in range(1, 32)] + [f"L23_V{i:03d}" for i in range(1, 26)]
)
